# Fundamental Quality Scorecard Using SEC + FMP Data

This Google Colab notebook builds a professional fundamental equity quality scorecard using real public-market data. It pulls company fundamentals from SEC EDGAR, optionally upgrades the dataset with Financial Modeling Prep if an API key is provided, adds market data from Yahoo Finance, adds macro/rate context from FRED, calculates institutional-style profitability, balance sheet, cash-flow quality, and growth-efficiency metrics, ranks companies, visualizes the results, and exports recruiter-friendly CSV files.

In [ ]:
%%capture
!pip -q install yfinance fredapi plotly seaborn tabulate tqdm

import os
import re
import json
import time
import math
import warnings
from datetime import datetime, timedelta
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display, Markdown

try:
    from google.colab import userdata
except Exception:
    userdata = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}" if isinstance(x, (int, float, np.floating)) else str(x))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

## Configuration

The notebook runs without an FMP key by using SEC EDGAR and Yahoo Finance. If you have API keys, add them in Colab using **Secrets** named `FMP_API_KEY`, `FRED_API_KEY`, and optionally `SEC_USER_AGENT`.

In [ ]:
def get_colab_secret(secret_name: str, default: str = "") -> str:
    try:
        if userdata is not None:
            value = userdata.get(secret_name)
            if value:
                return str(value)
    except Exception:
        pass
    return os.getenv(secret_name, default)


FMP_API_KEY = get_colab_secret("FMP_API_KEY", "")
FRED_API_KEY = get_colab_secret("FRED_API_KEY", "")
SEC_USER_AGENT = get_colab_secret(
    "SEC_USER_AGENT",
    "fundamental-quality-scorecard-colab/1.0 contact@example.com"
)

UNIVERSE_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA",
    "JPM", "BAC", "V", "MA", "UNH", "JNJ", "LLY", "PFE",
    "XOM", "CVX", "KO", "PEP", "WMT", "COST", "HD",
    "DIS", "NFLX", "ADBE", "CRM", "AMD", "INTC", "CAT", "BA"
]

BENCHMARK_TICKER = "SPY"
PRICE_HISTORY_PERIOD = "3y"
SEC_REQUEST_SLEEP_SECONDS = 0.12
MAX_SEC_RETRIES = 3
OUTPUT_DIR = "/content/fundamental_quality_scorecard_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Universe size: {len(UNIVERSE_TICKERS)} tickers")
print(f"FMP key detected: {bool(FMP_API_KEY)}")
print(f"FRED key detected: {bool(FRED_API_KEY)}")
print(f"Output folder: {OUTPUT_DIR}")

## Data Access + Utility Functions

In [ ]:
SEC_SESSION = requests.Session()
SEC_SESSION.headers.update({
    "User-Agent": SEC_USER_AGENT,
    "Accept": "application/json",
    "Accept-Encoding": "gzip, deflate",
})

GENERAL_SESSION = requests.Session()
GENERAL_SESSION.headers.update({
    "User-Agent": SEC_USER_AGENT,
    "Accept": "application/json",
})


def clean_symbol(symbol: str) -> str:
    return str(symbol).upper().strip().replace(".", "-")


def safe_float(value: Any) -> float:
    try:
        if value is None:
            return np.nan
        if isinstance(value, str) and value.strip() in {"", "None", "nan", "NaN"}:
            return np.nan
        return float(value)
    except Exception:
        return np.nan


def safe_divide(numerator: Any, denominator: Any) -> float:
    numerator = safe_float(numerator)
    denominator = safe_float(denominator)
    if np.isnan(numerator) or np.isnan(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator


def first_valid(*values: Any) -> Any:
    for value in values:
        if value is None:
            continue
        if isinstance(value, float) and np.isnan(value):
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        return value
    return np.nan


def format_large_number(value: Any) -> str:
    value = safe_float(value)
    if np.isnan(value):
        return "N/A"
    sign = "-" if value < 0 else ""
    value = abs(value)
    if value >= 1e12:
        return f"{sign}${value/1e12:,.2f}T"
    if value >= 1e9:
        return f"{sign}${value/1e9:,.2f}B"
    if value >= 1e6:
        return f"{sign}${value/1e6:,.2f}M"
    return f"{sign}${value:,.0f}"


def normalize_percent(value: Any) -> str:
    value = safe_float(value)
    if np.isnan(value):
        return "N/A"
    return f"{value * 100:,.1f}%"


def request_json(
    url: str,
    session: Optional[requests.Session] = None,
    params: Optional[Dict[str, Any]] = None,
    retries: int = 3,
    sleep_seconds: float = 0.25,
    timeout: int = 20,
) -> Optional[Any]:
    active_session = session or GENERAL_SESSION
    last_error = None

    for attempt in range(retries):
        try:
            response = active_session.get(url, params=params, timeout=timeout)
            if response.status_code == 429:
                time.sleep(sleep_seconds * (attempt + 2) * 3)
                continue
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(sleep_seconds * (attempt + 1))

    print(f"Request failed after {retries} tries: {url} | {last_error}")
    return None


def percentile_score(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if numeric.notna().sum() < 2:
        return pd.Series(np.nan, index=series.index)

    ranks = numeric.rank(pct=True, ascending=True)
    if not higher_is_better:
        ranks = 1 - ranks + (1 / numeric.notna().sum())
    return (ranks * 100).clip(0, 100)


def assign_quality_grade(score: Any) -> str:
    score = safe_float(score)
    if np.isnan(score):
        return "N/A"
    if score >= 90:
        return "A+"
    if score >= 80:
        return "A"
    if score >= 70:
        return "B"
    if score >= 60:
        return "C"
    if score >= 50:
        return "D"
    return "F"

## SEC EDGAR Fundamental Data Functions

In [ ]:
SEC_CONCEPTS = {
    "revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
    ],
    "gross_profit": ["GrossProfit"],
    "operating_income": ["OperatingIncomeLoss", "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"],
    "net_income": ["NetIncomeLoss", "ProfitLoss", "NetIncomeLossAvailableToCommonStockholdersBasic"],
    "assets": ["Assets"],
    "liabilities": ["Liabilities"],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "PartnersCapital",
    ],
    "current_assets": ["AssetsCurrent"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "cash": [
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        "Cash",
    ],
    "long_debt": [
        "LongTermDebtAndFinanceLeaseObligationsNoncurrent",
        "LongTermDebtNoncurrent",
        "LongTermDebt",
        "LongTermDebtAndFinanceLeaseObligations",
    ],
    "current_debt": [
        "CurrentDebt",
        "ShortTermBorrowings",
        "ShortTermDebt",
        "LongTermDebtAndFinanceLeaseObligationsCurrent",
        "CurrentPortionOfLongTermDebt",
    ],
    "operating_cash_flow": [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
    ],
    "capex": [
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "PaymentsToAcquireProductiveAssets",
        "CapitalExpenditures",
    ],
}


def load_sec_ticker_map() -> pd.DataFrame:
    url = "https://www.sec.gov/files/company_tickers.json"
    data = request_json(url, session=SEC_SESSION, retries=MAX_SEC_RETRIES, sleep_seconds=SEC_REQUEST_SLEEP_SECONDS)
    if not data:
        raise RuntimeError("Could not load SEC ticker map.")

    ticker_map = pd.DataFrame.from_dict(data, orient="index")
    ticker_map["ticker"] = ticker_map["ticker"].astype(str).str.upper()
    ticker_map["cik_str"] = ticker_map["cik_str"].astype(int).astype(str).str.zfill(10)
    ticker_map = ticker_map.rename(columns={"title": "sec_company_name"})
    return ticker_map.set_index("ticker")[["cik_str", "sec_company_name"]]


def fetch_sec_companyfacts(cik_str: str) -> Optional[Dict[str, Any]]:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_str}.json"
    data = request_json(
        url,
        session=SEC_SESSION,
        retries=MAX_SEC_RETRIES,
        sleep_seconds=SEC_REQUEST_SLEEP_SECONDS,
    )
    time.sleep(SEC_REQUEST_SLEEP_SECONDS)
    return data


def extract_sec_fact_records(
    companyfacts: Dict[str, Any],
    concept_candidates: List[str],
    unit_candidates: Iterable[str] = ("USD", "shares", "pure"),
    annual_only: bool = True,
) -> pd.DataFrame:
    if not companyfacts or "facts" not in companyfacts:
        return pd.DataFrame()

    us_gaap = companyfacts.get("facts", {}).get("us-gaap", {})
    records = []

    for concept_name in concept_candidates:
        concept_data = us_gaap.get(concept_name)
        if not concept_data:
            continue

        units = concept_data.get("units", {})
        for unit in unit_candidates:
            if unit not in units:
                continue

            for item in units.get(unit, []):
                form = str(item.get("form", ""))
                fp = str(item.get("fp", ""))
                value = safe_float(item.get("val"))

                if np.isnan(value):
                    continue

                if annual_only and form not in {"10-K", "10-K/A", "20-F", "20-F/A", "40-F", "40-F/A"}:
                    continue

                if annual_only and fp not in {"FY", ""}:
                    continue

                end_date = pd.to_datetime(item.get("end"), errors="coerce")
                start_date = pd.to_datetime(item.get("start"), errors="coerce")
                filed_date = pd.to_datetime(item.get("filed"), errors="coerce")

                duration_days = np.nan
                if pd.notna(end_date) and pd.notna(start_date):
                    duration_days = (end_date - start_date).days

                records.append({
                    "concept": concept_name,
                    "unit": unit,
                    "value": value,
                    "fy": item.get("fy"),
                    "fp": fp,
                    "form": form,
                    "start": start_date,
                    "end": end_date,
                    "filed": filed_date,
                    "duration_days": duration_days,
                })

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    if annual_only:
        instant_concepts = {"Assets", "Liabilities", "StockholdersEquity", "AssetsCurrent", "LiabilitiesCurrent"}
        annual_mask = df["duration_days"].isna() | (df["duration_days"].between(250, 460)) | df["concept"].isin(instant_concepts)
        df = df[annual_mask]
    df = df.dropna(subset=["end"])
    df = df.sort_values(["end", "filed"], ascending=[True, True])
    return df


def extract_latest_sec_value(
    companyfacts: Dict[str, Any],
    concept_candidates: List[str],
    unit_candidates: Iterable[str] = ("USD",),
    annual_only: bool = True,
) -> float:
    df = extract_sec_fact_records(companyfacts, concept_candidates, unit_candidates, annual_only=annual_only)
    if df.empty:
        return np.nan

    concept_priority = df.groupby("concept")["value"].count().sort_values(ascending=False)
    for concept_name in concept_priority.index.tolist():
        sub = df[df["concept"] == concept_name].copy()
        if not sub.empty:
            sub = sub.sort_values(["end", "filed"], ascending=[True, True])
            return safe_float(sub.iloc[-1]["value"])

    return np.nan


def extract_sec_annual_series(
    companyfacts: Dict[str, Any],
    concept_candidates: List[str],
    unit_candidates: Iterable[str] = ("USD",),
) -> pd.Series:
    df = extract_sec_fact_records(companyfacts, concept_candidates, unit_candidates, annual_only=True)
    if df.empty:
        return pd.Series(dtype=float)

    concept_counts = df.groupby("concept")["value"].count().sort_values(ascending=False)
    chosen_concept = concept_counts.index[0]
    sub = df[df["concept"] == chosen_concept].copy()
    sub = sub.sort_values(["end", "filed"]).drop_duplicates(subset=["end"], keep="last")
    series = sub.set_index("end")["value"].sort_index()
    return series


def fetch_sec_fundamentals_for_ticker(ticker: str, ticker_map: pd.DataFrame) -> Dict[str, Any]:
    ticker = clean_symbol(ticker)
    if ticker not in ticker_map.index:
        return {"ticker": ticker, "data_source_sec": False, "sec_error": "Ticker not found in SEC map"}

    cik_str = ticker_map.loc[ticker, "cik_str"]
    sec_company_name = ticker_map.loc[ticker, "sec_company_name"]
    facts = fetch_sec_companyfacts(cik_str)

    if not facts:
        return {
            "ticker": ticker,
            "cik": cik_str,
            "company_name_sec": sec_company_name,
            "data_source_sec": False,
            "sec_error": "SEC companyfacts unavailable",
        }

    values = {
        "ticker": ticker,
        "cik": cik_str,
        "company_name_sec": sec_company_name,
        "data_source_sec": True,
        "entity_name_sec": facts.get("entityName", sec_company_name),
    }

    for output_name, candidates in SEC_CONCEPTS.items():
        values[f"sec_{output_name}"] = extract_latest_sec_value(facts, candidates, unit_candidates=("USD",))

    revenue_series = extract_sec_annual_series(facts, SEC_CONCEPTS["revenue"], unit_candidates=("USD",))
    if len(revenue_series.dropna()) >= 2:
        values["sec_revenue_prior"] = revenue_series.dropna().iloc[-2]
        values["sec_revenue_latest_period"] = revenue_series.dropna().index[-1]
    else:
        values["sec_revenue_prior"] = np.nan
        values["sec_revenue_latest_period"] = pd.NaT

    long_debt = values.get("sec_long_debt", np.nan)
    current_debt = values.get("sec_current_debt", np.nan)
    values["sec_total_debt"] = np.nansum([long_debt, current_debt])
    if np.isnan(safe_float(long_debt)) and np.isnan(safe_float(current_debt)):
        values["sec_total_debt"] = np.nan

    capex_raw = values.get("sec_capex", np.nan)
    cfo = values.get("sec_operating_cash_flow", np.nan)
    values["sec_capex_abs"] = abs(capex_raw) if not np.isnan(safe_float(capex_raw)) else np.nan
    values["sec_free_cash_flow"] = cfo - values["sec_capex_abs"] if not np.isnan(safe_float(cfo)) else np.nan

    return values

## Financial Modeling Prep + Yahoo Finance + FRED Functions

In [ ]:
def fmp_get_json(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Optional[Any]:
    if not FMP_API_KEY:
        return None

    base_url = "https://financialmodelingprep.com/api/v3"
    final_params = dict(params or {})
    final_params["apikey"] = FMP_API_KEY
    url = f"{base_url}/{endpoint.lstrip('/')}"

    return request_json(url, session=GENERAL_SESSION, params=final_params, retries=2, sleep_seconds=0.5)


def fetch_fmp_fundamentals_for_ticker(ticker: str) -> Dict[str, Any]:
    ticker = clean_symbol(ticker)
    if not FMP_API_KEY:
        return {"ticker": ticker, "data_source_fmp": False, "fmp_error": "No FMP API key provided"}

    income = fmp_get_json(f"income-statement/{ticker}", {"limit": 5})
    balance = fmp_get_json(f"balance-sheet-statement/{ticker}", {"limit": 5})
    cashflow = fmp_get_json(f"cash-flow-statement/{ticker}", {"limit": 5})
    profile = fmp_get_json(f"profile/{ticker}", {})

    if not isinstance(income, list) or not income:
        return {"ticker": ticker, "data_source_fmp": False, "fmp_error": "FMP statements unavailable"}

    income_latest = income[0] if isinstance(income, list) and income else {}
    income_prior = income[1] if isinstance(income, list) and len(income) > 1 else {}
    balance_latest = balance[0] if isinstance(balance, list) and balance else {}
    cashflow_latest = cashflow[0] if isinstance(cashflow, list) and cashflow else {}
    profile_latest = profile[0] if isinstance(profile, list) and profile else {}

    total_debt = first_valid(
        balance_latest.get("totalDebt"),
        safe_float(balance_latest.get("shortTermDebt")) + safe_float(balance_latest.get("longTermDebt"))
    )

    capex = safe_float(cashflow_latest.get("capitalExpenditure"))
    cfo = safe_float(cashflow_latest.get("netCashProvidedByOperatingActivities"))
    capex_abs = abs(capex) if not np.isnan(capex) else np.nan
    free_cash_flow = cfo - capex_abs if not np.isnan(cfo) else np.nan

    return {
        "ticker": ticker,
        "data_source_fmp": True,
        "company_name_fmp": profile_latest.get("companyName"),
        "sector_fmp": profile_latest.get("sector"),
        "industry_fmp": profile_latest.get("industry"),
        "market_cap_fmp": safe_float(profile_latest.get("mktCap")),
        "price_fmp": safe_float(profile_latest.get("price")),
        "beta_fmp": safe_float(profile_latest.get("beta")),
        "fmp_revenue": safe_float(income_latest.get("revenue")),
        "fmp_revenue_prior": safe_float(income_prior.get("revenue")),
        "fmp_gross_profit": safe_float(income_latest.get("grossProfit")),
        "fmp_operating_income": safe_float(income_latest.get("operatingIncome")),
        "fmp_net_income": safe_float(income_latest.get("netIncome")),
        "fmp_assets": safe_float(balance_latest.get("totalAssets")),
        "fmp_liabilities": safe_float(balance_latest.get("totalLiabilities")),
        "fmp_equity": safe_float(balance_latest.get("totalStockholdersEquity")),
        "fmp_current_assets": safe_float(balance_latest.get("totalCurrentAssets")),
        "fmp_current_liabilities": safe_float(balance_latest.get("totalCurrentLiabilities")),
        "fmp_cash": safe_float(balance_latest.get("cashAndCashEquivalents")),
        "fmp_total_debt": safe_float(total_debt),
        "fmp_operating_cash_flow": cfo,
        "fmp_capex_abs": capex_abs,
        "fmp_free_cash_flow": free_cash_flow,
        "fmp_fiscal_date": income_latest.get("date"),
    }


def fetch_yfinance_metadata_for_ticker(ticker: str) -> Dict[str, Any]:
    ticker = clean_symbol(ticker)
    output = {"ticker": ticker, "data_source_yfinance": False}

    try:
        yf_ticker = yf.Ticker(ticker)
        info = {}
        try:
            info = yf_ticker.get_info() or {}
        except Exception:
            info = getattr(yf_ticker, "info", {}) or {}

        fast_info = {}
        try:
            fast_info = dict(yf_ticker.fast_info)
        except Exception:
            fast_info = {}

        output.update({
            "data_source_yfinance": True,
            "company_name_yf": first_valid(info.get("longName"), info.get("shortName")),
            "sector_yf": info.get("sector"),
            "industry_yf": info.get("industry"),
            "market_cap_yf": safe_float(first_valid(info.get("marketCap"), fast_info.get("market_cap"))),
            "price_yf": safe_float(first_valid(info.get("currentPrice"), fast_info.get("last_price"))),
            "beta_yf": safe_float(info.get("beta")),
            "currency_yf": info.get("currency"),
            "exchange_yf": info.get("exchange"),
        })
    except Exception as exc:
        output["yf_error"] = str(exc)

    return output


def fetch_fred_series(series_id: str, observation_start: str = "2000-01-01") -> pd.Series:
    series_id = series_id.upper().strip()

    if FRED_API_KEY:
        url = "https://api.stlouisfed.org/fred/series/observations"
        params = {
            "series_id": series_id,
            "api_key": FRED_API_KEY,
            "file_type": "json",
            "observation_start": observation_start,
        }
        data = request_json(url, session=GENERAL_SESSION, params=params, retries=2)
        if data and "observations" in data:
            df = pd.DataFrame(data["observations"])
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df[series_id] = pd.to_numeric(df["value"].replace(".", np.nan), errors="coerce")
            return df.dropna(subset=["date"]).set_index("date")[series_id].dropna()

    try:
        csv_url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        df = pd.read_csv(csv_url)
        df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
        df[series_id] = pd.to_numeric(df[series_id].replace(".", np.nan), errors="coerce")
        df = df[df["observation_date"] >= pd.to_datetime(observation_start)]
        return df.dropna(subset=["observation_date"]).set_index("observation_date")[series_id].dropna()
    except Exception as exc:
        print(f"FRED fetch failed for {series_id}: {exc}")
        return pd.Series(dtype=float, name=series_id)


def combine_fundamental_sources(sec_row: Dict[str, Any], fmp_row: Dict[str, Any], yf_row: Dict[str, Any]) -> Dict[str, Any]:
    ticker = clean_symbol(sec_row.get("ticker") or fmp_row.get("ticker") or yf_row.get("ticker"))

    combined = {
        "ticker": ticker,
        "company_name": first_valid(
            fmp_row.get("company_name_fmp"),
            yf_row.get("company_name_yf"),
            sec_row.get("entity_name_sec"),
            sec_row.get("company_name_sec"),
        ),
        "sector": first_valid(fmp_row.get("sector_fmp"), yf_row.get("sector_yf"), "Unknown"),
        "industry": first_valid(fmp_row.get("industry_fmp"), yf_row.get("industry_yf"), "Unknown"),
        "market_cap": first_valid(fmp_row.get("market_cap_fmp"), yf_row.get("market_cap_yf")),
        "price": first_valid(fmp_row.get("price_fmp"), yf_row.get("price_yf")),
        "beta_reported": first_valid(fmp_row.get("beta_fmp"), yf_row.get("beta_yf")),
        "currency": yf_row.get("currency_yf"),
        "exchange": yf_row.get("exchange_yf"),
        "cik": sec_row.get("cik"),
        "has_sec_data": bool(sec_row.get("data_source_sec")),
        "has_fmp_data": bool(fmp_row.get("data_source_fmp")),
        "has_yfinance_data": bool(yf_row.get("data_source_yfinance")),
    }

    fields = [
        "revenue",
        "revenue_prior",
        "gross_profit",
        "operating_income",
        "net_income",
        "assets",
        "liabilities",
        "equity",
        "current_assets",
        "current_liabilities",
        "cash",
        "total_debt",
        "operating_cash_flow",
        "capex_abs",
        "free_cash_flow",
    ]

    for field in fields:
        combined[field] = first_valid(fmp_row.get(f"fmp_{field}"), sec_row.get(f"sec_{field}"))

    combined["fundamental_source_priority"] = "FMP + SEC" if combined["has_fmp_data"] and combined["has_sec_data"] else (
        "FMP" if combined["has_fmp_data"] else ("SEC" if combined["has_sec_data"] else "Market metadata only")
    )

    return combined

## Pull Fundamentals for the Universe

In [ ]:
def build_fundamental_dataset(tickers: List[str]) -> pd.DataFrame:
    ticker_map = load_sec_ticker_map()
    rows = []

    for ticker in tqdm([clean_symbol(t) for t in tickers], desc="Fetching company fundamentals"):
        sec_row = fetch_sec_fundamentals_for_ticker(ticker, ticker_map)
        fmp_row = fetch_fmp_fundamentals_for_ticker(ticker)
        yf_row = fetch_yfinance_metadata_for_ticker(ticker)
        combined = combine_fundamental_sources(sec_row, fmp_row, yf_row)
        rows.append(combined)

    df = pd.DataFrame(rows)

    numeric_cols = [
        "market_cap", "price", "beta_reported", "revenue", "revenue_prior",
        "gross_profit", "operating_income", "net_income", "assets", "liabilities",
        "equity", "current_assets", "current_liabilities", "cash", "total_debt",
        "operating_cash_flow", "capex_abs", "free_cash_flow",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


fundamentals_raw = build_fundamental_dataset(UNIVERSE_TICKERS)

display(Markdown(f"### Raw Fundamental Data Pulled: {len(fundamentals_raw)} companies"))
display(
    fundamentals_raw[
        [
            "ticker", "company_name", "sector", "market_cap", "revenue",
            "net_income", "assets", "equity", "free_cash_flow",
            "fundamental_source_priority"
        ]
    ].head(15)
)

## Calculate Fundamental Metrics

In [ ]:
def add_fundamental_metrics(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    result["revenue_growth_yoy"] = result.apply(lambda r: safe_divide(r["revenue"] - r["revenue_prior"], r["revenue_prior"]), axis=1)
    result["gross_margin"] = result.apply(lambda r: safe_divide(r["gross_profit"], r["revenue"]), axis=1)
    result["operating_margin"] = result.apply(lambda r: safe_divide(r["operating_income"], r["revenue"]), axis=1)
    result["net_margin"] = result.apply(lambda r: safe_divide(r["net_income"], r["revenue"]), axis=1)
    result["return_on_assets"] = result.apply(lambda r: safe_divide(r["net_income"], r["assets"]), axis=1)
    result["return_on_equity"] = result.apply(lambda r: safe_divide(r["net_income"], r["equity"]), axis=1)
    result["asset_turnover"] = result.apply(lambda r: safe_divide(r["revenue"], r["assets"]), axis=1)

    result["debt_to_equity"] = result.apply(lambda r: safe_divide(r["total_debt"], r["equity"]), axis=1)
    result["liabilities_to_assets"] = result.apply(lambda r: safe_divide(r["liabilities"], r["assets"]), axis=1)
    result["current_ratio"] = result.apply(lambda r: safe_divide(r["current_assets"], r["current_liabilities"]), axis=1)
    result["cash_to_debt"] = result.apply(lambda r: safe_divide(r["cash"], r["total_debt"]), axis=1)
    result["cash_to_assets"] = result.apply(lambda r: safe_divide(r["cash"], r["assets"]), axis=1)

    result["fcf_margin"] = result.apply(lambda r: safe_divide(r["free_cash_flow"], r["revenue"]), axis=1)
    result["fcf_to_net_income"] = result.apply(lambda r: safe_divide(r["free_cash_flow"], r["net_income"]), axis=1)
    result["cfo_to_net_income"] = result.apply(lambda r: safe_divide(r["operating_cash_flow"], r["net_income"]), axis=1)
    result["accruals_ratio"] = result.apply(lambda r: safe_divide(r["net_income"] - r["operating_cash_flow"], r["assets"]), axis=1)

    bounded_cols = [
        "return_on_equity", "debt_to_equity", "cash_to_debt",
        "fcf_to_net_income", "cfo_to_net_income"
    ]
    for col in bounded_cols:
        result[col] = pd.to_numeric(result[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        result[col] = result[col].clip(lower=-5, upper=5)

    return result


fundamentals = add_fundamental_metrics(fundamentals_raw)

metric_preview_cols = [
    "ticker", "company_name", "sector", "revenue_growth_yoy", "gross_margin",
    "operating_margin", "net_margin", "return_on_assets", "return_on_equity",
    "debt_to_equity", "current_ratio", "fcf_margin", "accruals_ratio"
]

display(Markdown("### Calculated Fundamental Ratios"))
display(fundamentals[metric_preview_cols].head(15))

## Pull Market Prices + Macro Context

In [ ]:
def download_price_matrix(tickers: List[str], period: str = "3y") -> pd.DataFrame:
    tickers = sorted(set([clean_symbol(t) for t in tickers]))
    raw = yf.download(tickers, period=period, auto_adjust=True, progress=False, threads=True)

    if raw.empty:
        raise RuntimeError("Yahoo Finance returned no price data.")

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"].copy()
        elif "Close" in raw.columns.get_level_values(1):
            close = raw.xs("Close", level=1, axis=1).copy()
        else:
            raise RuntimeError("Could not locate Close prices in yfinance result.")
    else:
        close = raw[["Close"]].rename(columns={"Close": tickers[0]}).copy()

    close = close.dropna(axis=1, how="all")
    close.index = pd.to_datetime(close.index)
    return close


def calculate_market_features(price_matrix: pd.DataFrame, benchmark_ticker: str = "SPY") -> pd.DataFrame:
    close = price_matrix.copy()
    returns = close.pct_change().dropna(how="all")

    rows = []
    benchmark_returns = returns[benchmark_ticker].dropna() if benchmark_ticker in returns.columns else None

    for ticker in close.columns:
        if ticker == benchmark_ticker:
            continue

        prices = close[ticker].dropna()
        asset_returns = returns[ticker].dropna() if ticker in returns.columns else pd.Series(dtype=float)

        if len(prices) < 60 or len(asset_returns) < 60:
            rows.append({"ticker": ticker})
            continue

        latest_price = prices.iloc[-1]
        return_3m = safe_divide(latest_price, prices.iloc[-63]) - 1 if len(prices) > 63 else np.nan
        return_6m = safe_divide(latest_price, prices.iloc[-126]) - 1 if len(prices) > 126 else np.nan
        return_12m = safe_divide(latest_price, prices.iloc[-252]) - 1 if len(prices) > 252 else np.nan

        vol_1y = asset_returns.tail(252).std() * np.sqrt(252)
        downside = asset_returns.tail(252)
        downside_vol_1y = downside[downside < 0].std() * np.sqrt(252)

        rolling_max = prices.tail(252).cummax()
        drawdown = prices.tail(252) / rolling_max - 1
        max_drawdown_1y = drawdown.min()

        beta = np.nan
        if benchmark_returns is not None:
            aligned = pd.concat([asset_returns, benchmark_returns], axis=1, join="inner").dropna()
            aligned.columns = ["asset", "benchmark"]
            if len(aligned) > 100 and aligned["benchmark"].var() > 0:
                beta = aligned["asset"].cov(aligned["benchmark"]) / aligned["benchmark"].var()

        rows.append({
            "ticker": ticker,
            "return_3m": return_3m,
            "return_6m": return_6m,
            "return_12m": return_12m,
            "volatility_1y": vol_1y,
            "downside_volatility_1y": downside_vol_1y,
            "max_drawdown_1y": max_drawdown_1y,
            "beta_vs_benchmark": beta,
            "price_latest_yf": latest_price,
        })

    return pd.DataFrame(rows)


all_price_tickers = sorted(set(UNIVERSE_TICKERS + [BENCHMARK_TICKER]))
price_matrix = download_price_matrix(all_price_tickers, PRICE_HISTORY_PERIOD)
market_features = calculate_market_features(price_matrix, BENCHMARK_TICKER)

dgs10 = fetch_fred_series("DGS10", observation_start="2015-01-01")
fed_funds = fetch_fred_series("FEDFUNDS", observation_start="2015-01-01")
credit_spread = fetch_fred_series("BAA10Y", observation_start="2015-01-01")

macro_dates = [s.dropna().index[-1] for s in [dgs10, fed_funds, credit_spread] if len(s.dropna())]
macro_snapshot = {
    "latest_10y_treasury_yield_pct": dgs10.dropna().iloc[-1] if len(dgs10.dropna()) else np.nan,
    "latest_fed_funds_rate_pct": fed_funds.dropna().iloc[-1] if len(fed_funds.dropna()) else np.nan,
    "latest_baa_10y_credit_spread_pct": credit_spread.dropna().iloc[-1] if len(credit_spread.dropna()) else np.nan,
    "macro_snapshot_date": str(max(macro_dates).date()) if macro_dates else "N/A",
}

display(Markdown("### Market Feature Preview"))
display(market_features.head(15))

display(Markdown("### FRED Macro Snapshot"))
display(pd.DataFrame([macro_snapshot]).T.rename(columns={0: "value"}))

## Build the Quality Score

The model scores each company across four institutional-style categories: profitability, balance sheet strength, cash-flow quality, and growth efficiency. Each metric becomes a percentile score, then group scores are blended into one 0-100 quality score.

In [ ]:
def build_quality_scorecard(fundamentals_df: pd.DataFrame, market_df: pd.DataFrame) -> pd.DataFrame:
    df = fundamentals_df.merge(market_df, on="ticker", how="left")

    metric_direction = {
        "gross_margin": True,
        "operating_margin": True,
        "net_margin": True,
        "return_on_assets": True,
        "return_on_equity": True,
        "current_ratio": True,
        "cash_to_debt": True,
        "cash_to_assets": True,
        "debt_to_equity": False,
        "liabilities_to_assets": False,
        "fcf_margin": True,
        "fcf_to_net_income": True,
        "cfo_to_net_income": True,
        "accruals_ratio": False,
        "revenue_growth_yoy": True,
        "asset_turnover": True,
        "volatility_1y": False,
        "downside_volatility_1y": False,
        "max_drawdown_1y": True,
    }

    for metric, higher_is_better in metric_direction.items():
        if metric in df.columns:
            df[f"{metric}_score"] = percentile_score(df[metric], higher_is_better=higher_is_better)

    groups = {
        "profitability_score": [
            "gross_margin_score", "operating_margin_score", "net_margin_score",
            "return_on_assets_score", "return_on_equity_score"
        ],
        "balance_sheet_score": [
            "current_ratio_score", "cash_to_debt_score", "cash_to_assets_score",
            "debt_to_equity_score", "liabilities_to_assets_score"
        ],
        "cash_flow_quality_score": [
            "fcf_margin_score", "fcf_to_net_income_score", "cfo_to_net_income_score", "accruals_ratio_score"
        ],
        "growth_efficiency_score": [
            "revenue_growth_yoy_score", "asset_turnover_score"
        ],
        "market_risk_score": [
            "volatility_1y_score", "downside_volatility_1y_score", "max_drawdown_1y_score"
        ],
    }

    for group_name, score_cols in groups.items():
        existing_cols = [col for col in score_cols if col in df.columns]
        df[group_name] = df[existing_cols].mean(axis=1, skipna=True)

    group_weights = {
        "profitability_score": 0.35,
        "balance_sheet_score": 0.25,
        "cash_flow_quality_score": 0.25,
        "growth_efficiency_score": 0.10,
        "market_risk_score": 0.05,
    }

    weighted_scores = []
    for _, row in df.iterrows():
        numerator = 0.0
        denominator = 0.0
        for group_name, weight in group_weights.items():
            value = safe_float(row.get(group_name))
            if not np.isnan(value):
                numerator += value * weight
                denominator += weight
        weighted_scores.append(numerator / denominator if denominator else np.nan)

    df["quality_score"] = weighted_scores
    df["quality_grade"] = df["quality_score"].apply(assign_quality_grade)
    df["overall_rank"] = df["quality_score"].rank(ascending=False, method="min")
    df["sector_quality_rank"] = df.groupby("sector")["quality_score"].rank(ascending=False, method="min")
    df["sector_quality_percentile"] = df.groupby("sector")["quality_score"].rank(pct=True, ascending=True) * 100

    df["market_cap_bucket"] = pd.cut(
        df["market_cap"],
        bins=[-np.inf, 2e9, 10e9, 50e9, 200e9, np.inf],
        labels=["Micro/Small", "Small/Mid", "Mid/Large", "Mega", "Titan"],
    )

    df = df.sort_values("quality_score", ascending=False).reset_index(drop=True)
    return df


scorecard = build_quality_scorecard(fundamentals, market_features)

display(Markdown("### Fundamental Quality Scorecard: Top 15"))
top_display_cols = [
    "overall_rank", "ticker", "company_name", "sector", "quality_score", "quality_grade",
    "profitability_score", "balance_sheet_score", "cash_flow_quality_score",
    "growth_efficiency_score", "market_risk_score", "market_cap",
    "gross_margin", "return_on_equity", "debt_to_equity", "fcf_margin",
]
display(scorecard[top_display_cols].head(15))

## Professional Summary Tables

In [ ]:
def make_display_table(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    cols = [
        "overall_rank", "ticker", "company_name", "sector", "quality_score", "quality_grade",
        "profitability_score", "balance_sheet_score", "cash_flow_quality_score",
        "growth_efficiency_score", "market_risk_score", "market_cap",
        "revenue", "gross_margin", "operating_margin", "return_on_equity",
        "debt_to_equity", "current_ratio", "fcf_margin", "return_12m", "volatility_1y",
    ]
    table = df[cols].head(n).copy()

    for col in [
        "quality_score", "profitability_score", "balance_sheet_score",
        "cash_flow_quality_score", "growth_efficiency_score", "market_risk_score"
    ]:
        table[col] = table[col].map(lambda x: f"{x:,.1f}" if pd.notna(x) else "N/A")

    for col in ["gross_margin", "operating_margin", "return_on_equity", "fcf_margin", "return_12m", "volatility_1y"]:
        table[col] = table[col].map(normalize_percent)

    table["market_cap"] = table["market_cap"].map(format_large_number)
    table["revenue"] = table["revenue"].map(format_large_number)
    table["debt_to_equity"] = table["debt_to_equity"].map(lambda x: f"{x:,.2f}x" if pd.notna(x) else "N/A")
    table["current_ratio"] = table["current_ratio"].map(lambda x: f"{x:,.2f}x" if pd.notna(x) else "N/A")
    return table


top_15_table = make_display_table(scorecard, 15)
bottom_10_table = make_display_table(scorecard.sort_values("quality_score", ascending=True), 10)

display(Markdown("### Top 15 Quality Companies"))
display(top_15_table)

display(Markdown("### Bottom 10 Quality Companies in This Universe"))
display(bottom_10_table)

## Visualizations: Rankings, Sector Quality, and Financial Drivers

In [ ]:
plot_df = scorecard.dropna(subset=["quality_score"]).copy()

fig = px.bar(
    plot_df.head(20).sort_values("quality_score", ascending=True),
    x="quality_score",
    y="ticker",
    orientation="h",
    color="sector",
    hover_data=["company_name", "quality_grade", "profitability_score", "balance_sheet_score", "cash_flow_quality_score"],
    title="Top Companies by Fundamental Quality Score",
    labels={"quality_score": "Quality Score", "ticker": "Ticker"},
)
fig.update_layout(height=650, template="plotly_white", legend_title_text="Sector")
fig.show()


sector_summary = (
    scorecard
    .groupby("sector", dropna=False)
    [["quality_score", "profitability_score", "balance_sheet_score", "cash_flow_quality_score", "growth_efficiency_score", "market_risk_score"]]
    .mean()
    .sort_values("quality_score", ascending=False)
)

fig = px.imshow(
    sector_summary.T,
    text_auto=".1f",
    aspect="auto",
    title="Average Quality Components by Sector",
    labels=dict(x="Sector", y="Score Component", color="Average Score"),
)
fig.update_layout(height=550, template="plotly_white")
fig.show()


scatter_df = scorecard.copy()
scatter_df["market_cap_bubble"] = scatter_df["market_cap"].fillna(scorecard["market_cap"].median())
scatter_df["market_cap_bubble"] = scatter_df["market_cap_bubble"].clip(lower=1e9)

fig = px.scatter(
    scatter_df,
    x="debt_to_equity",
    y="return_on_equity",
    size="market_cap_bubble",
    color="quality_score",
    hover_name="ticker",
    hover_data=["company_name", "sector", "quality_grade", "gross_margin", "fcf_margin"],
    title="Quality Map: Return on Equity vs. Debt-to-Equity",
    labels={
        "debt_to_equity": "Debt / Equity",
        "return_on_equity": "Return on Equity",
        "quality_score": "Quality Score",
    },
    size_max=45,
)
fig.update_layout(height=650, template="plotly_white")
fig.show()

## Metric Correlation Heatmap + Quality Component Diagnostics

In [ ]:
core_metric_cols = [
    "quality_score", "gross_margin", "operating_margin", "net_margin",
    "return_on_assets", "return_on_equity", "debt_to_equity",
    "liabilities_to_assets", "current_ratio", "cash_to_assets",
    "fcf_margin", "fcf_to_net_income", "cfo_to_net_income",
    "accruals_ratio", "revenue_growth_yoy", "asset_turnover",
    "return_12m", "volatility_1y", "max_drawdown_1y",
]

corr_df = scorecard[[col for col in core_metric_cols if col in scorecard.columns]].apply(pd.to_numeric, errors="coerce")
corr_matrix = corr_df.corr(min_periods=5)

plt.figure(figsize=(15, 10))
sns.heatmap(corr_matrix, annot=False, cmap="vlag", center=0, linewidths=0.4)
plt.title("Correlation Heatmap: Fundamental, Market, and Quality Metrics", fontsize=16)
plt.tight_layout()
plt.show()


component_cols = [
    "profitability_score",
    "balance_sheet_score",
    "cash_flow_quality_score",
    "growth_efficiency_score",
    "market_risk_score",
]

component_df = scorecard[["ticker", "quality_score"] + component_cols].head(15).copy()
component_long = component_df.melt(
    id_vars=["ticker", "quality_score"],
    value_vars=component_cols,
    var_name="component",
    value_name="score",
)

fig = px.bar(
    component_long,
    x="ticker",
    y="score",
    color="component",
    barmode="group",
    title="Quality Score Component Breakdown: Top 15",
    labels={"score": "Component Score", "ticker": "Ticker", "component": "Component"},
)
fig.update_layout(height=650, template="plotly_white", legend_title_text="")
fig.show()

## Single-Company Tear Sheet

Change `SELECTED_TICKER` to inspect any company in the scored universe.

In [ ]:
SELECTED_TICKER = scorecard.iloc[0]["ticker"]

def display_company_tearsheet(scorecard_df: pd.DataFrame, selected_ticker: str) -> None:
    selected_ticker = clean_symbol(selected_ticker)
    if selected_ticker not in set(scorecard_df["ticker"]):
        raise ValueError(f"{selected_ticker} is not in the scorecard universe.")

    row = scorecard_df[scorecard_df["ticker"] == selected_ticker].iloc[0]

    display(Markdown(f'''
### {row["ticker"]} — {row["company_name"]}

**Sector:** {row["sector"]}  
**Industry:** {row["industry"]}  
**Quality Score:** {row["quality_score"]:.1f} / 100  
**Quality Grade:** {row["quality_grade"]}  
**Overall Rank:** #{int(row["overall_rank"])} out of {len(scorecard_df)}  
**Fundamental Source:** {row["fundamental_source_priority"]}  
    '''.strip()))

    key_stats = pd.DataFrame({
        "Metric": [
            "Market Cap", "Revenue", "Net Income", "Free Cash Flow",
            "Gross Margin", "Operating Margin", "Net Margin",
            "ROA", "ROE", "Debt / Equity", "Current Ratio",
            "FCF Margin", "Accruals Ratio", "12M Return", "1Y Volatility",
        ],
        "Value": [
            format_large_number(row.get("market_cap")),
            format_large_number(row.get("revenue")),
            format_large_number(row.get("net_income")),
            format_large_number(row.get("free_cash_flow")),
            normalize_percent(row.get("gross_margin")),
            normalize_percent(row.get("operating_margin")),
            normalize_percent(row.get("net_margin")),
            normalize_percent(row.get("return_on_assets")),
            normalize_percent(row.get("return_on_equity")),
            f"{row.get('debt_to_equity'):,.2f}x" if pd.notna(row.get("debt_to_equity")) else "N/A",
            f"{row.get('current_ratio'):,.2f}x" if pd.notna(row.get("current_ratio")) else "N/A",
            normalize_percent(row.get("fcf_margin")),
            normalize_percent(row.get("accruals_ratio")),
            normalize_percent(row.get("return_12m")),
            normalize_percent(row.get("volatility_1y")),
        ],
    })
    display(key_stats)

    radar_categories = [
        "Profitability",
        "Balance Sheet",
        "Cash Flow Quality",
        "Growth Efficiency",
        "Market Risk",
    ]
    radar_values = [
        row.get("profitability_score"),
        row.get("balance_sheet_score"),
        row.get("cash_flow_quality_score"),
        row.get("growth_efficiency_score"),
        row.get("market_risk_score"),
    ]

    radar_values = [0 if pd.isna(v) else float(v) for v in radar_values]
    radar_values_closed = radar_values + [radar_values[0]]
    radar_categories_closed = radar_categories + [radar_categories[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=radar_values_closed,
        theta=radar_categories_closed,
        fill="toself",
        name=selected_ticker,
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        showlegend=False,
        title=f"{selected_ticker} Quality Component Radar",
        template="plotly_white",
        height=550,
    )
    fig.show()


display_company_tearsheet(scorecard, SELECTED_TICKER)

## Illustrative Basket Performance

This is **not** a true historical factor backtest because the notebook uses the latest available fundamentals to form baskets. Treat it as an illustrative visualization of how the current high-quality and low-quality baskets have behaved historically.

In [ ]:
def calculate_illustrative_basket_performance(
    scorecard_df: pd.DataFrame,
    price_df: pd.DataFrame,
    benchmark_ticker: str,
    quantile: float = 0.25,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    scored = scorecard_df.dropna(subset=["quality_score"]).copy()
    n = max(3, int(len(scored) * quantile))

    high_quality = scored.sort_values("quality_score", ascending=False).head(n)["ticker"].tolist()
    low_quality = scored.sort_values("quality_score", ascending=True).head(n)["ticker"].tolist()

    available_high = [t for t in high_quality if t in price_df.columns]
    available_low = [t for t in low_quality if t in price_df.columns]

    returns = price_df.pct_change().dropna(how="all")

    result = pd.DataFrame(index=returns.index)
    result["High Quality Basket"] = returns[available_high].mean(axis=1) if available_high else np.nan
    result["Low Quality Basket"] = returns[available_low].mean(axis=1) if available_low else np.nan
    if benchmark_ticker in returns.columns:
        result["Benchmark"] = returns[benchmark_ticker]

    cumulative = (1 + result.fillna(0)).cumprod()

    holdings = pd.DataFrame({
        "high_quality_basket": pd.Series(available_high),
        "low_quality_basket": pd.Series(available_low),
    })

    return cumulative, holdings


basket_cumulative, basket_holdings = calculate_illustrative_basket_performance(scorecard, price_matrix, BENCHMARK_TICKER)

display(Markdown("### Basket Constituents"))
display(basket_holdings)

fig = go.Figure()
for column in basket_cumulative.columns:
    fig.add_trace(go.Scatter(
        x=basket_cumulative.index,
        y=basket_cumulative[column],
        mode="lines",
        name=column,
    ))

fig.update_layout(
    title="Illustrative Historical Performance of Current Quality Baskets",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    template="plotly_white",
    height=600,
)
fig.show()


basket_returns = basket_cumulative.pct_change().dropna()
performance_rows = []
for col in basket_returns.columns:
    r = basket_returns[col].dropna()
    if len(r) == 0:
        continue
    annual_return = (basket_cumulative[col].iloc[-1] / basket_cumulative[col].iloc[0]) ** (252 / len(r)) - 1
    annual_vol = r.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol and not np.isnan(annual_vol) else np.nan
    max_dd = (basket_cumulative[col] / basket_cumulative[col].cummax() - 1).min()
    performance_rows.append({
        "Basket": col,
        "Annualized Return": annual_return,
        "Annualized Volatility": annual_vol,
        "Sharpe Approx.": sharpe,
        "Max Drawdown": max_dd,
    })

performance_table = pd.DataFrame(performance_rows)
display(Markdown("### Illustrative Performance Metrics"))
display(performance_table)

## Export Portfolio-Ready Outputs

In [ ]:
scorecard_output_path = os.path.join(OUTPUT_DIR, "fundamental_quality_scorecard.csv")
top_15_output_path = os.path.join(OUTPUT_DIR, "top_15_quality_companies.csv")
performance_output_path = os.path.join(OUTPUT_DIR, "illustrative_basket_performance.csv")
macro_output_path = os.path.join(OUTPUT_DIR, "macro_snapshot.json")

scorecard.to_csv(scorecard_output_path, index=False)
top_15_table.to_csv(top_15_output_path, index=False)
basket_cumulative.to_csv(performance_output_path)

with open(macro_output_path, "w") as f:
    json.dump(macro_snapshot, f, indent=2)

export_summary = pd.DataFrame({
    "File": [
        scorecard_output_path,
        top_15_output_path,
        performance_output_path,
        macro_output_path,
    ],
    "Description": [
        "Full raw + scored company dataset",
        "Formatted top 15 quality table",
        "Illustrative current-basket performance time series",
        "FRED macro/rate snapshot",
    ],
})

display(Markdown("### Exported Files"))
display(export_summary)

try:
    from google.colab import files
    print("To download manually, open the file browser on the left or run:")
    print(f"files.download('{scorecard_output_path}')")
except Exception:
    pass

## Final Summary

This notebook created an end-to-end **Fundamental Quality Scorecard** using SEC EDGAR, optional Financial Modeling Prep data, Yahoo Finance market data, and FRED macro context. It calculated profitability, balance sheet, cash-flow quality, growth-efficiency, and market-risk metrics, converted them into institutional-style percentile scores, visualized rankings and factor drivers, built a company tear sheet, created illustrative quality-basket performance, and exported clean CSV files.

**Resume-ready bullet:** Built a Python-based fundamental equity quality research platform in Google Colab using SEC EDGAR, FMP, Yahoo Finance, and FRED data to rank public companies by profitability, balance sheet strength, cash-flow quality, growth efficiency, and market risk, with professional visualizations and exportable investment scorecards.